This code defines the file paths for the vegetable image dataset stored in Google Drive. The variable `data_dir` contains the main folder location where all the dataset images are saved. Then, `train_dir`, `val_dir`, and `test_dir` create paths for the training, validation, and testing folders by adding their respective folder names (`/train`, `/validation`, and `/test`) to the main dataset path. These directories are later used to load images for training, validating, and testing the machine learning or deep learning model.


In [ ]:
data_dir = "/content/drive/MyDrive/Assessment_Last/Vegetable Images"

train_dir = data_dir + "/train"
val_dir = data_dir + "/validation"
test_dir = data_dir + "/test"

This code imports the libraries required for building and training a deep learning model for image classification. The `os` and `time` modules are used for file handling and measuring execution time, while `numpy` helps with numerical operations. `matplotlib.pyplot` is used to visualize images and graphs. From TensorFlow Keras, `ImageDataGenerator` is imported for image preprocessing and augmentation, `layers` and `models` are used to build the neural network, and `Adam` and `SGD` are optimization algorithms used during training. `VGG16` is a pre-trained convolutional neural network model used for transfer learning, and `Model` helps customize or modify the architecture. Finally, `classification_report` from Scikit-learn is used to evaluate the model’s performance by showing metrics such as precision, recall, and F1-score.


In [16]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model

from sklearn.metrics import classification_report

**Data Generators with Augmentation**

This code prepares the image dataset for training, validation, and testing in a deep learning model. The variable `img_size = (224, 224)` sets all images to a size of 224×224 pixels, while `batch_size = 32` means the model processes 32 images at a time during training. `ImageDataGenerator` is used to preprocess the images by scaling pixel values between 0 and 1 using `rescale=1./255`. For the training data, additional augmentation techniques such as rotation, zooming, and horizontal flipping are applied to increase dataset variety and improve model performance. The `flow_from_directory()` function loads images directly from the train, validation, and test folders, automatically assigning labels based on folder names. The training and validation data are shuffled, while the test data uses `shuffle=False` to maintain the correct order for evaluation. Finally, `train_data.num_classes` stores and prints the total number of vegetable classes in the dataset.


In [17]:
img_size = (224, 224)
batch_size = 32

# Augmentation ONLY for training
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

val_data = val_gen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

test_data = test_gen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

num_classes = train_data.num_classes
print("Classes:", num_classes)

Found 15000 images belonging to 15 classes.
Found 3000 images belonging to 15 classes.
Found 3000 images belonging to 15 classes.
Classes: 15


The output shows that the dataset was successfully loaded from the directories. The training dataset contains 15,000 images, the validation dataset contains 3,000 images, and the test dataset also contains 3,000 images. All the images are divided into 15 different vegetable classes. Finally, `Classes: 15` confirms that the model will classify images into 15 categories.


**Baseline Model**

Model Architecture

This code builds and compiles a baseline Convolutional Neural Network (CNN) model for vegetable image classification using Keras Sequential API. The model contains three convolutional layers with 32, 64, and 128 filters that extract important image features, and each convolutional layer is followed by a max-pooling layer to reduce image dimensions and computation. The `Flatten()` layer converts the extracted 2D feature maps into a 1D vector so it can be passed to fully connected dense layers. Three dense layers with 128, 64, and 32 neurons are used to learn complex patterns from the extracted features. The final dense layer uses the `softmax` activation function to classify images into the 15 vegetable classes. The model is compiled using the `Adam` optimizer, `categorical_crossentropy` loss function for multi-class classification, and `accuracy` as the evaluation metric. Finally, `model_baseline.summary()` displays the architecture and parameters of the model.


In [18]:
model_baseline = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),

    layers.Dense(num_classes, activation='softmax')
])

model_baseline.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_baseline.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 15)             │           495 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,179,791 (42.65 MB)

 Trainable params: 11,179,791 (42.65 MB)

 Non-trainable params: 0 (0.00 B)

The output shows the summary of the baseline CNN model architecture. At the beginning, Keras gives a warning suggesting that instead of directly passing input shape to the first convolutional layer, it is better practice to use an input shape layer in Sequential models. The model contains three convolutional layers Conv2D that extract image features and three max-pooling layers MaxPooling2D that reduce the spatial dimensions of the feature maps. After feature extraction, the Flatten layer converts the 2D feature maps into a one-dimensional vector of size 86,528. This flattened output is passed through fully connected dense layers with 128, 64, and 32 neurons for learning complex patterns. The final dense layer has 15 neurons, corresponding to the 15 vegetable classes, and uses the softmax activation function for classification.

The model has a total of 11,179,791 trainable parameters, which require approximately 42.65 MB of memory. Most of the parameters are present in the first dense layer because it connects the large flattened feature vector to 128 neurons. Since all parameters are trainable and there are no non-trainable parameters, the entire network will be updated during training.


Train Baseline

This code trains the baseline CNN model using the training dataset and validates it using the validation dataset. The time.time() function is used to record the starting time before training begins. The fit() function trains the model for 15 epochs, where one epoch means the model processes the entire training dataset once. During training, the model learns patterns from the training images and evaluates its performance on the validation dataset after each epoch. The training history, including accuracy and loss values, is stored in history_baseline. After training is completed, the total training time is calculated by subtracting the starting time from the ending time and stored in baseline_time. Finally, the total training time is printed.


In [19]:
start = time.time()

history_baseline = model_baseline.fit(
    train_data,
    validation_data=val_data,
    epochs=15
)

baseline_time = time.time() - start
print("Baseline Training Time:", baseline_time)

Epoch 1/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 176s 367ms/step - accuracy: 0.3444 - loss: 1.9471 - val_accuracy: 0.7767 - val_loss: 0.7313
Epoch 2/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 172s 367ms/step - accuracy: 0.7505 - loss: 0.7694 - val_accuracy: 0.8250 - val_loss: 0.5809
Epoch 3/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 172s 367ms/step - accuracy: 0.8551 - loss: 0.4655 - val_accuracy: 0.8847 - val_loss: 0.3981
Epoch 4/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 174s 372ms/step - accuracy: 0.9054 - loss: 0.2986 - val_accuracy: 0.8653 - val_loss: 0.4586
Epoch 5/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 176s 375ms/step - accuracy: 0.9218 - loss: 0.2324 - val_accuracy: 0.9373 - val_loss: 0.2207
Epoch 6/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 177s 377ms/step - accuracy: 0.9440 - loss: 0.1778 - val_accuracy: 0.9537 - val_loss: 0.1541
Epoch 7/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 172s 366ms/step - accuracy: 0.9480 - loss: 0.1562 - val_accuracy: 0.9307 - val_loss: 0.2589
Epoch 8/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 173s 368ms/step - accuracy: 0.9567 -

The training results show that the CNN model’s performance improved consistently over 15 epochs. In the first epoch, the model started with a training accuracy of 34.44% and a validation accuracy of 77.67%, with a high loss value of 1.9471, indicating that the model was still learning basic image features. As the epochs increased, the training accuracy steadily improved while the loss decreased, showing that the model was learning patterns effectively from the vegetable images.

By the middle epochs, the model achieved over 90% training accuracy and validation accuracy, demonstrating strong feature extraction and classification capability. Although there were small fluctuations in validation accuracy and loss during some epochs, the overall trend showed continuous improvement and better generalization.

In the final epoch, the model achieved a training accuracy of 97.74% and a validation accuracy of 98.00% with a very low validation loss of 0.0842. This indicates that the CNN model performed extremely well in classifying the 15 vegetable categories with minimal prediction error. The total training time for the baseline model was approximately 2671.88 seconds, or around 44.5 minutes.


Plot Loss and Accuracy

This code evaluates the trained baseline CNN model on the test dataset and measures its performance on unseen data. The `evaluate()` function returns the test loss and accuracy, where accuracy shows how well the model correctly classifies the test images. Then, `model_baseline.predict(test_data)` is used to generate prediction probabilities for each test image, and `np.argmax()` converts these probabilities into final class labels by selecting the highest probability class. Finally, `classification_report()` compares the true labels with the predicted labels and provides detailed metrics such as precision, recall, F1-score, and support for all 15 vegetable classes, giving a complete view of the model’s performance across each category.


In [21]:
loss, acc = model_baseline.evaluate(test_data)
print("Baseline Accuracy:", acc)

y_pred = np.argmax(model_baseline.predict(test_data), axis=1)
print(classification_report(test_data.classes, y_pred))

94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 96ms/step - accuracy: 0.9889 - loss: 0.0436
Baseline Accuracy: 0.9826666712760925
94/94 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       200
           1       0.99      0.99      0.99       200
           2       0.99      0.99      0.99       200
           3       0.99      0.99      0.99       200
           4       0.94      0.99      0.96       200
           5       0.98      0.95      0.97       200
           6       0.99      1.00      0.99       200
           7       0.99      0.99      0.99       200
           8       0.94      0.98      0.96       200
           9       0.99      0.99      0.99       200
          10       0.99      0.97      0.98       200
          11       1.00      0.99      0.99       200
          12       0.99      0.94      0.96       200
          13       1.00      0.99      0.99       200
          14       0.97      0.97    

The output shows the complete pipeline results of the baseline CNN model for vegetable image classification. The dataset contains 15,000 training images, 3,000 validation images, and 3,000 test images across 15 classes. The model summary indicates a CNN architecture with convolutional, pooling, and dense layers, totaling about 11.18 million trainable parameters. During training over 15 epochs, the model steadily improved from low initial accuracy (34.44%) to high performance, reaching about 97.74% training accuracy and 98.00% validation accuracy, while the loss consistently decreased, showing effective learning. The total training time was approximately 2671.88 seconds. After evaluation on the test dataset, the model achieved a test accuracy of about 98.27%, confirming strong generalization on unseen data. The classification report further shows high precision, recall, and F1-scores (mostly around 0.98–0.99) across all 15 classes, indicating that the model performs very well and correctly classifies most vegetable images with minimal errors.


Evaluation

This code evaluates the trained baseline CNN model on the test dataset and measures its final performance on unseen data. The `evaluate()` function computes the test loss and accuracy, where `baseline_acc` represents how correctly the model classifies the test images overall. Next, `model_baseline.predict(test_data)` generates prediction probabilities for each test image, and `np.argmax()` converts these probabilities into final class labels by selecting the highest probability for each sample. Finally, `classification_report()` compares the true labels with the predicted labels and provides detailed evaluation metrics such as precision, recall, F1-score, and support for all 15 vegetable classes, giving a complete breakdown of how well the model performs for each category.


In [22]:
baseline_loss, baseline_acc = model_baseline.evaluate(test_data)
print("Baseline Accuracy:", baseline_acc)

y_pred = np.argmax(model_baseline.predict(test_data), axis=1)
print(classification_report(test_data.classes, y_pred))

94/94 ━━━━━━━━━━━━━━━━━━━━ 6s 63ms/step - accuracy: 0.9889 - loss: 0.0436
Baseline Accuracy: 0.9826666712760925
94/94 ━━━━━━━━━━━━━━━━━━━━ 6s 59ms/step
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       200
           1       0.99      0.99      0.99       200
           2       0.99      0.99      0.99       200
           3       0.99      0.99      0.99       200
           4       0.94      0.99      0.96       200
           5       0.98      0.95      0.97       200
           6       0.99      1.00      0.99       200
           7       0.99      0.99      0.99       200
           8       0.94      0.98      0.96       200
           9       0.99      0.99      0.99       200
          10       0.99      0.97      0.98       200
          11       1.00      0.99      0.99       200
          12       0.99      0.94      0.96       200
          13       1.00      0.99      0.99       200
          14       0.97      0.97    

This output shows the final evaluation results of the baseline CNN model on the test dataset. The model achieved a test accuracy of about **98.27%** with a very low loss of **0.0436**, indicating strong performance on unseen images. The classification report further breaks down the results for all 15 vegetable classes, showing consistently high precision, recall, and F1-scores (mostly between 0.96 and 0.99), which means the model is correctly identifying most classes with very few misclassifications. Overall, both the accuracy score and the detailed metrics confirm that the model generalizes well and performs reliably across all categories.


**Deeper Model**

Architecture with Regularization

This code builds a deeper and more advanced CNN model for vegetable image classification. Compared to the baseline model, it has additional convolutional layers (32, 64, 128, and 256 filters) to extract more complex and detailed features from the images. Each convolutional layer is followed by **Batch Normalization**, which helps stabilize and speed up training, and **MaxPooling** layers, which reduce spatial dimensions and prevent overfitting. After feature extraction, the model uses a `Flatten` layer to convert data into a 1D vector, followed by fully connected dense layers with 256 and 128 neurons. **Dropout layers (0.5 and 0.3)** are added to reduce overfitting by randomly deactivating neurons during training. The final layer uses **softmax activation** to classify images into the 15 vegetable classes. The model is compiled using the Adam optimizer, categorical cross-entropy loss, and accuracy as the evaluation metric, and `model_deep.summary()` displays the full architecture and number of parameters.


In [23]:
model_deep = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),

    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(num_classes, activation='softmax')
])

model_deep.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_deep.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 222, 222, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 109, 109, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 52, 52, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 24, 24, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 24, 24, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 12, 12, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │     9,437,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 15)             │         1,935 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,862,607 (37.62 MB)

 Trainable params: 9,861,647 (37.62 MB)

 Non-trainable params: 960 (3.75 KB)

This output shows the architecture summary of your deep CNN model. It contains four convolutional layers (32, 64, 128, and 256 filters) that progressively extract more complex features from the images, and each of them is followed by batch normalization, which helps stabilize and speed up training. Max-pooling layers reduce the spatial size of feature maps step by step, helping to reduce computation and control overfitting.

After feature extraction, the output is flattened into a 36,864-dimensional vector and passed through fully connected dense layers with 256 and 128 neurons. Dropout layers are applied between them to randomly deactivate neurons during training, which further helps prevent overfitting and improves generalization. The final output layer has 15 neurons, corresponding to the 15 vegetable classes, using softmax activation for classification.

Overall, the model has about **9.86 million trainable parameters**, which is slightly smaller than the baseline model despite being deeper, due to better architectural design using batch normalization and dropout for efficiency and regularization.


Training

This code trains the deep CNN model using the training dataset and evaluates it on the validation dataset for 15 epochs. The `time.time()` function is used to record the starting time before training begins. The `fit()` function then trains the model, where it learns features from the training images and checks performance on the validation data after each epoch. The training history, including accuracy and loss values for both training and validation sets, is stored in `history_deep`. After training is completed, the total time taken is calculated by subtracting the start time from the end time and stored in `deep_time`. Finally, the total training time of the deep model is printed to compare its efficiency with other models.


In [ ]:
start = time.time()

history_deep = model_deep.fit(
    train_data,
    validation_data=val_data,
    epochs=15
)

deep_time = time.time() - start
print("Deep Model Training Time:", deep_time)

Epoch 1/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 214s 439ms/step - accuracy: 0.1634 - loss: 5.3976 - val_accuracy: 0.2753 - val_loss: 2.3178
Epoch 2/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 199s 424ms/step - accuracy: 0.2228 - loss: 2.4446 - val_accuracy: 0.2877 - val_loss: 2.2203
Epoch 3/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 196s 417ms/step - accuracy: 0.2845 - loss: 2.1987 - val_accuracy: 0.4283 - val_loss: 1.7743
Epoch 4/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 194s 414ms/step - accuracy: 0.3387 - loss: 2.0100 - val_accuracy: 0.5240 - val_loss: 1.4954
Epoch 5/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 193s 411ms/step - accuracy: 0.4065 - loss: 1.7779 - val_accuracy: 0.4393 - val_loss: 1.6732
Epoch 6/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 191s 407ms/step - accuracy: 0.4300 - loss: 1.7051 - val_accuracy: 0.4883 - val_loss: 1.8828
Epoch 7/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 193s 412ms/step - accuracy: 0.4665 - loss: 1.5744 - val_accuracy: 0.3427 - val_loss: 2.6256
Epoch 8/15
469/469 ━━━━━━━━━━━━━━━━━━━━ 201s 428ms/step - accuracy: 0.5120 -

The training results show the performance of the deeper CNN model over 15 epochs. In the first epoch, the model starts with a low training accuracy of about 16.34% and a high loss of 5.3976, while the validation accuracy is 27.53%, indicating that the model is still learning basic patterns from the data. As training progresses, both training accuracy and validation accuracy gradually improve, showing that the model is learning better feature representations through its deeper architecture with batch normalization and dropout.

By mid-training, the model reaches around 63% training accuracy, and validation accuracy improves significantly, peaking at about 83% in later epochs. However, there are some fluctuations in validation performance (for example, drops in certain epochs), which suggests some instability or overfitting during training.

In the final epoch, the model achieves approximately 76.43% training accuracy and 78.97% validation accuracy with a validation loss of 0.6478. This shows that while the deep model learns useful features, it does not perform as well as the baseline CNN in this case. The total training time for the deep model is about 2914.30 seconds (around 48.5 minutes), which is longer than the baseline model due to its increased complexity.


Evaluation

This code evaluates the trained deep CNN model on the test dataset and measures its final performance on unseen images. The `evaluate()` function computes the test loss and accuracy, where `deep_acc` represents how well the model classifies the test images overall. Next, `model_deep.predict(test_data)` generates probability predictions for each test image, and `np.argmax()` converts these probabilities into final predicted class labels by selecting the class with the highest probability. Finally, `classification_report()` compares the true labels with the predicted labels and provides detailed metrics such as precision, recall, F1-score, and support for all 15 vegetable classes, giving a complete breakdown of how well the deep model performs for each category.


In [25]:
deep_loss, deep_acc = model_deep.evaluate(test_data)
print("Deep Model Accuracy:", deep_acc)

y_pred = np.argmax(model_deep.predict(test_data), axis=1)
print(classification_report(test_data.classes, y_pred))

94/94 ━━━━━━━━━━━━━━━━━━━━ 10s 109ms/step - accuracy: 0.8083 - loss: 0.7049
Deep Model Accuracy: 0.7806666493415833
94/94 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step
              precision    recall  f1-score   support

           0       0.96      0.77      0.85       200
           1       0.99      0.90      0.94       200
           2       0.45      0.69      0.55       200
           3       0.49      0.87      0.63       200
           4       0.75      0.96      0.84       200
           5       0.60      0.82      0.69       200
           6       0.92      0.98      0.95       200
           7       0.97      0.92      0.94       200
           8       0.97      0.45      0.61       200
           9       0.85      0.91      0.87       200
          10       0.52      0.06      0.10       200
          11       0.92      0.94      0.93       200
          12       0.92      0.92      0.92       200
          13       0.89      0.85      0.87       200
          14       0.95      0.67

The output shows the final evaluation of the deep CNN model on the test dataset. The model achieved a test accuracy of about **78.07%** with a loss of **0.7049**, which indicates moderate performance on unseen data. After prediction, the classification report provides a detailed breakdown of results for all 15 vegetable classes.

From the report, we can see that the model performs very well on some classes (for example classes 1, 6, 7, and 11 with precision and recall around 0.90–0.99), but it performs poorly on others such as class 10, which has very low recall (0.06), meaning it fails to correctly identify most images in that class. This imbalance in performance suggests that the model struggles with certain categories and is not as stable as the baseline model.

Overall, even though the deep model is more complex, its accuracy (78%) is lower than the baseline CNN (98%), showing that increased depth does not always guarantee better performance and may lead to overfitting or training instability in this case.


Comparison

This code compares the performance and efficiency of the baseline CNN model and the deep CNN model. It prints the accuracy of both models on the test dataset (`baseline_acc` and `deep_acc`) so that their classification performance can be directly compared. It also prints the total training time for each model (`baseline_time` and `deep_time`) to show how long each model took to train. This helps evaluate the trade-off between accuracy and computational cost, allowing you to clearly see which model performs better and which one is more efficient in terms of training time.


In [26]:
print("Baseline Accuracy:", baseline_acc)
print("Deep Accuracy:", deep_acc)

print("Baseline Time:", baseline_time)
print("Deep Time:", deep_time)

Baseline Accuracy: 0.9826666712760925
Deep Accuracy: 0.7806666493415833
Baseline Time: 2671.8814692497253
Deep Time: 2914.298819065094


The results clearly show a comparison between the baseline CNN and the deep CNN model in terms of accuracy and training time.

The **baseline model performs significantly better**, achieving a test accuracy of **98.27%**, compared to the deep model’s **78.07%**, meaning the simpler architecture generalizes better on this dataset. In terms of training time, the baseline model took about **2671.88 seconds**, while the deep model took longer at **2914.30 seconds**, indicating that the deeper architecture requires more computation but does not improve performance in this case.

Overall, this comparison shows that a more complex model is not always better—here, the baseline CNN is both **more accurate and more efficient** than the deeper CNN.


Optimizer Analysis (SGD vs Adam)

SGD

This code re-compiles the deep CNN model using the **SGD (Stochastic Gradient Descent)** optimizer with a learning rate of **0.01**, instead of the previously used Adam optimizer. The loss function remains **categorical cross-entropy**, which is suitable for multi-class classification, and accuracy is used as the evaluation metric. After compiling, the model is trained again for **10 epochs** using the training dataset, while its performance is validated on the validation dataset after each epoch. The training history, including accuracy and loss values for both training and validation sets, is stored in `history_sgd`, which can later be used to analyze how SGD affects model performance compared to Adam.


In [27]:
model_deep.compile(
    optimizer=SGD(learning_rate=0.01),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_sgd = model_deep.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 208s 433ms/step - accuracy: 0.7659 - loss: 0.6907 - val_accuracy: 0.6733 - val_loss: 1.6181
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 186s 397ms/step - accuracy: 0.7786 - loss: 0.6254 - val_accuracy: 0.8657 - val_loss: 0.5051
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 184s 392ms/step - accuracy: 0.7881 - loss: 0.6150 - val_accuracy: 0.8673 - val_loss: 0.4108
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 195s 416ms/step - accuracy: 0.7899 - loss: 0.5958 - val_accuracy: 0.8973 - val_loss: 0.3047
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 197s 420ms/step - accuracy: 0.7979 - loss: 0.5658 - val_accuracy: 0.8977 - val_loss: 0.3524
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 192s 409ms/step - accuracy: 0.7995 - loss: 0.5610 - val_accuracy: 0.8970 - val_loss: 0.3097
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 194s 413ms/step - accuracy: 0.7971 - loss: 0.5552 - val_accuracy: 0.5923 - val_loss: 2.8274
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 195s 416ms/step - accuracy: 0.8066 -

The training results show the performance of the deep CNN model after switching to the **SGD optimizer** for 10 epochs. At the beginning, the model achieves a relatively high training accuracy of about **76.59%**, but validation accuracy is lower (**67.33%**) with a high loss, indicating unstable learning at the start. As training progresses, the model gradually improves and reaches around **80% training accuracy**, with validation accuracy peaking around **89–90%** in several epochs, showing that SGD helps the model converge more steadily over time.

However, there is some instability in later epochs, especially at **Epoch 7**, where validation accuracy drops sharply to **59.23%** and loss spikes to **2.8274**, which suggests the model struggles with consistency. Despite this, the model recovers in subsequent epochs and ends with about **80.79% training accuracy** and **88.40% validation accuracy**.

Overall, using SGD improves the performance of the deep model compared to its previous training, but the results still show fluctuations, meaning the model is sensitive to optimization settings and may require further tuning for better stability.


Adam

This code re-compiles the deep CNN model using the **Adam optimizer** with a learning rate of **0.001**, which is commonly used because it adapts the learning rate automatically during training for faster and more stable convergence. The loss function remains **categorical cross-entropy**, suitable for multi-class classification, and accuracy is used as the evaluation metric. After compiling, the model is trained again for **10 epochs** using the training dataset, while its performance is validated on the validation dataset after each epoch. The training results are stored in `history_adam`, which can later be used to analyze and compare how Adam performs against SGD and other optimizers in terms of accuracy, loss, and stability.


In [28]:
model_deep.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_adam = model_deep.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 209s 430ms/step - accuracy: 0.7936 - loss: 0.6120 - val_accuracy: 0.8160 - val_loss: 0.8231
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 194s 413ms/step - accuracy: 0.8042 - loss: 0.5965 - val_accuracy: 0.7790 - val_loss: 0.8958
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 190s 405ms/step - accuracy: 0.8087 - loss: 0.5935 - val_accuracy: 0.8393 - val_loss: 0.5568
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 189s 402ms/step - accuracy: 0.8264 - loss: 0.4961 - val_accuracy: 0.8207 - val_loss: 0.5829
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 192s 409ms/step - accuracy: 0.8306 - loss: 0.5088 - val_accuracy: 0.9043 - val_loss: 0.2977
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 194s 413ms/step - accuracy: 0.8372 - loss: 0.4675 - val_accuracy: 0.8230 - val_loss: 0.4764
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 192s 410ms/step - accuracy: 0.8448 - loss: 0.4478 - val_accuracy: 0.9090 - val_loss: 0.2807
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 192s 409ms/step - accuracy: 0.8528 -

The training results show the performance of the deep CNN model after switching to the **Adam optimizer** for 10 epochs. At the beginning, the model achieves around **79.36% training accuracy** with a validation accuracy of **81.60%**, but the validation loss is relatively high, indicating some instability. As training continues, the model gradually improves its learning, with training accuracy increasing steadily to about **86.27%** by the final epoch.

The validation performance fluctuates across epochs—reaching a peak of about **90.90% accuracy in Epoch 7**, where the model also achieves a low validation loss of **0.2807**, showing its best generalization performance. However, in later epochs, validation accuracy drops slightly and loss increases again, suggesting mild overfitting or instability toward the end of training.

Overall, Adam provides smoother and faster convergence compared to SGD, achieving better peak validation performance, but the model still shows some variation in later epochs, meaning further tuning (like regularization or early stopping) could improve stability.


**Transfer Learning**

Load Model

In [29]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

Preprocessing Change

This code defines image data generators for training, validation, and testing using Keras’ `ImageDataGenerator`. The `train_gen` applies data augmentation techniques such as rotation (up to 20 degrees), zooming (up to 20%), and horizontal flipping to increase dataset diversity and help the model generalize better. All datasets use `preprocess_input`, which standardizes the images according to the requirements of a pre-trained model (such as VGG16 or similar), ensuring the input data is properly scaled and normalized. Unlike the training set, the validation and test generators only apply preprocessing without augmentation, so their images remain unchanged for fair evaluation of the model’s performance.


In [32]:
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

Load Pretrained Model

This code loads the **MobileNetV2** model pre-trained on the **ImageNet dataset** to use it for transfer learning. The parameter `weights='imagenet'` means the model already has learned useful features from a large dataset of images. `include_top=False` removes the original classification layer so that the model can be customized for a new task (in this case, vegetable classification). The `input_shape=(224,224,3)` specifies that the input images should be 224×224 pixels with 3 color channels (RGB). This base model will act as a feature extractor, where it learns and reuses powerful image features for the new dataset instead of training from scratch.


In [33]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


This output shows that TensorFlow is downloading the **pre-trained weights** for the **MobileNetV2** model from Google’s storage server. The file `mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5` contains the learned weights from training on the **ImageNet dataset**. Since `include_top=False` was used, only the convolutional base (feature extractor part) is downloaded, not the final classification layers.

Once downloaded, these weights allow the model to instantly use powerful pre-learned image features, saving training time and improving performance through transfer learning.


Freeze Base Model (Feature Extraction)

This code freezes all the layers of the **MobileNetV2 base model** by setting `layer.trainable = False` for each layer. This means the weights of the pre-trained model will not be updated during training. By doing this, the model retains the useful features learned from the **ImageNet dataset** and only allows the new classification layers (added on top of MobileNetV2) to be trained on the vegetable dataset. This helps reduce training time, prevents overfitting, and makes the model more stable, especially when working with a smaller dataset.


In [35]:
for layer in base_model.layers:
    layer.trainable = False

Add Custom Classification Layers

This code builds a **transfer learning model using MobileNetV2** by adding custom classification layers on top of the pre-trained base model. First, `base_model.output` is taken as the starting point, which contains extracted high-level image features. Then, `GlobalAveragePooling2D()` is applied to reduce these feature maps into a single vector by averaging each feature channel, making the model more efficient.

After that, a **Dense layer with 128 neurons** and ReLU activation is added to learn task-specific patterns, followed by a **Dropout layer (0.5)** to reduce overfitting by randomly turning off half of the neurons during training. Finally, a **Dense output layer with softmax activation** is added with `num_classes` neurons to classify images into different vegetable categories. The final model is created by combining the original MobileNetV2 input with these new layers using the `Model()` API, forming a complete transfer learning architecture.


In [38]:
from tensorflow.keras import layers, Model

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

output = layers.Dense(num_classes, activation='softmax')(x)

model_mobilenet = Model(inputs=base_model.input, outputs=output)

Compile Model

This code compiles and summarizes the **MobileNetV2-based transfer learning model**. The model is compiled using the **Adam optimizer**, which helps adjust learning rates automatically for efficient training. The loss function used is **categorical cross-entropy**, which is suitable for multi-class classification problems like vegetable image classification. The metric chosen is **accuracy**, which measures how many predictions are correctly classified. Finally, `model_mobilenet.summary()` displays the full architecture of the model, including the frozen MobileNetV2 base layers and the newly added classification head, along with the total number of parameters and how many are trainable.


In [39]:
model_mobilenet.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_mobilenet.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_4[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,423,887 (9.25 MB)

 Trainable params: 165,903 (648.06 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

You first built a deep convolutional neural network from scratch with multiple convolution layers, batch normalization, max pooling, and dense layers. Although the model was powerful, it had a very large number of parameters, especially after the Flatten layer, which led to overfitting and poor generalization. As a result, even though training accuracy improved, the test accuracy stayed lower (around 0.78) and validation performance was unstable. You then tried different optimizers, where SGD performed better than Adam because it gave more stable updates and reduced overfitting, leading to improved validation accuracy. After that, you used MobileNetV2 for transfer learning, which significantly improved efficiency because it was already pretrained on ImageNet and could extract useful features like edges and textures without learning from scratch. In this model, the total parameters were about 2.42 million, but only around 165,903 were trainable while the rest were frozen pretrained weights. This made the model faster, more stable, and better at generalization, since only the final classification layers were updated for your 15 classes instead of retraining the entire network.


Train (Feature Extraction Phase)

This code trains your MobileNetV2 transfer learning model for 10 epochs using the training and validation datasets while also measuring the total training time. First, the starting time is recorded before training begins using `time.time()`. Then the model is trained with `model_mobilenet.fit()`, where MobileNetV2 uses pretrained features and only updates the newly added classification layers, making the training more efficient. After training is completed, the ending time is recorded and subtracted from the starting time to calculate the total training duration, which is then printed. This helps you compare how fast MobileNetV2 trains compared to your deep CNN models and shows the advantage of transfer learning in reducing training time while maintaining good performance.


In [40]:
start = time.time()

history_mn = model_mobilenet.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

mn_time = time.time() - start
print("MobileNetV2 Training Time:", mn_time)

Epoch 1/10


2026-05-06 11:24:18.164898: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-06 11:24:18.303302: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


426/469 ━━━━━━━━━━━━━━━━━━━━ 17s 407ms/step - accuracy: 0.8097 - loss: 0.6466

2026-05-06 11:27:23.658174: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-06 11:27:23.801398: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-06 11:27:23.938344: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


469/469 ━━━━━━━━━━━━━━━━━━━━ 238s 469ms/step - accuracy: 0.8201 - loss: 0.6117 - val_accuracy: 0.9927 - val_loss: 0.0246
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 194s 413ms/step - accuracy: 0.9771 - loss: 0.0776 - val_accuracy: 0.9957 - val_loss: 0.0169
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 185s 395ms/step - accuracy: 0.9844 - loss: 0.0515 - val_accuracy: 0.9967 - val_loss: 0.0106
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 186s 396ms/step - accuracy: 0.9877 - loss: 0.0408 - val_accuracy: 0.9983 - val_loss: 0.0061
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 187s 399ms/step - accuracy: 0.9888 - loss: 0.0345 - val_accuracy: 0.9987 - val_loss: 0.0057
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 186s 396ms/step - accuracy: 0.9895 - loss: 0.0349 - val_accuracy: 0.9977 - val_loss: 0.0095
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 192s 409ms/step - accuracy: 0.9916 - loss: 0.0256 - val_accuracy: 0.9990 - val_loss: 0.0064
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 191s 408ms/step - accuracy: 0.9924 - loss: 0.02

Your MobileNetV2 model shows very strong performance during training and validation, reaching around 99% accuracy on both training and validation sets after just a few epochs, which indicates that transfer learning is highly effective for your dataset. In the first epoch, the model quickly improves because it uses pretrained features from ImageNet, and only the final classification layers are being trained. This leads to fast convergence, very low loss values, and consistently high validation accuracy (up to 0.999). However, the warning messages about CUDA kernel timing are related to GPU measurement delays and do not affect the actual training results. Overall, the model trains in about 1932 seconds and achieves near-perfect accuracy, showing that MobileNetV2 is significantly more efficient and accurate compared to your deep CNN trained from scratch.


**Fine-Tuning**

This code is used for **fine-tuning your MobileNetV2 model**. Initially, you had frozen all layers of the pretrained base model so that only the final classification layers were trained. Now, by setting `layer.trainable = True` for the last 20 layers, you are unfreezing the top part of MobileNetV2 so it can adapt more specifically to your dataset. These last layers are responsible for learning higher-level, task-specific features, so allowing them to update improves accuracy and model performance while still keeping most of the pretrained knowledge intact. This approach balances transfer learning and custom learning, making the model more accurate without retraining the entire network from scratch.


In [41]:
for layer in base_model.layers[-20:]:
    layer.trainable = True

**Recompile with LOW LR**

This code recompiles your MobileNetV2 model for **fine-tuning** using the Adam optimizer with a very low learning rate of `1e-5`. After unfreezing the last 20 layers of the pretrained base model, you need a smaller learning rate so that the already learned ImageNet features are not destroyed during training. Instead, the model makes only small adjustments to those layers to better fit your dataset while keeping most of the useful pretrained knowledge intact. The model continues to use categorical crossentropy as the loss function since it is a multi-class classification problem, and accuracy is used as the evaluation metric to monitor performance during training.


In [42]:
from tensorflow.keras.optimizers import Adam

model_mobilenet.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

Fine-Tune Training

This code continues training your MobileNetV2 model using **fine-tuning**, where the last few layers of the pretrained base model have been unfrozen and are now being updated along with the newly added classification layers. The model is trained for 5 epochs using the training data, while also evaluating its performance on the validation set after each epoch to monitor improvement. Since the learning rate was set very low (1e-5), the model makes small and careful updates to the pretrained weights, allowing it to better adapt to your specific dataset without losing the useful features it learned from ImageNet. This stage usually helps improve accuracy and generalization after initial transfer learning training.


In [43]:
history_ft = model_mobilenet.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 235s 470ms/step - accuracy: 0.9122 - loss: 0.3332 - val_accuracy: 0.9973 - val_loss: 0.0086
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 197s 420ms/step - accuracy: 0.9749 - loss: 0.0876 - val_accuracy: 0.9973 - val_loss: 0.0081
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 198s 421ms/step - accuracy: 0.9803 - loss: 0.0624 - val_accuracy: 0.9990 - val_loss: 0.0056
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 195s 416ms/step - accuracy: 0.9850 - loss: 0.0474 - val_accuracy: 0.9983 - val_loss: 0.0056
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 190s 404ms/step - accuracy: 0.9877 - loss: 0.0412 - val_accuracy: 0.9983 - val_loss: 0.0049


During fine-tuning of your MobileNetV2 model, you trained it for 5 epochs after unfreezing the last few layers with a very low learning rate. The results show that the model quickly achieves very high performance, starting with about 91% training accuracy in the first epoch and improving steadily up to nearly 99% by the final epoch. The validation accuracy remains extremely high throughout training, reaching up to 99.9%, while the validation loss stays very low and continues to decrease, indicating strong generalization and minimal overfitting. This improvement happens because fine-tuning allows the pretrained layers to slightly adjust to your specific dataset, refining high-level features while preserving the useful knowledge learned from ImageNet, leading to faster convergence and better overall performance compared to training from scratch.


Evaluation

This code evaluates your fine-tuned MobileNetV2 model on the test dataset and measures its final performance. First, `model_mobilenet.evaluate(test_data)` calculates the overall loss and accuracy on unseen test data, giving you a clear measure of how well the model generalizes beyond training and validation sets. Then, the model predicts class probabilities for the test images using `model_mobilenet.predict(test_data)`, and `np.argmax()` converts these probabilities into final class labels. Finally, `classification_report()` compares the predicted labels with the true labels from `test_data.classes` and provides detailed metrics such as precision, recall, and F1-score for each class. This helps you understand not only the overall accuracy but also how well the model performs on each individual category.


In [44]:
loss, acc = model_mobilenet.evaluate(test_data)
print("MobileNetV2 Accuracy:", acc)

import numpy as np
from sklearn.metrics import classification_report

y_pred = np.argmax(model_mobilenet.predict(test_data), axis=1)
print(classification_report(test_data.classes, y_pred))

94/94 ━━━━━━━━━━━━━━━━━━━━ 10s 106ms/step - accuracy: 0.9992 - loss: 0.0069
MobileNetV2 Accuracy: 0.9993333220481873
94/94 ━━━━━━━━━━━━━━━━━━━━ 14s 102ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       200
           1       0.99      0.99      0.99       200
           2       1.00      1.00      1.00       200
           3       1.00      1.00      1.00       200
           4       1.00      1.00      1.00       200
           5       1.00      1.00      1.00       200
           6       1.00      1.00      1.00       200
           7       1.00      1.00      1.00       200
           8       1.00      1.00      1.00       200
           9       1.00      1.00      1.00       200
          10       1.00      1.00      1.00       200
          11       1.00      1.00      1.00       200
          12       1.00      0.99      1.00       200
          13       1.00      1.00      1.00       200
          14       1.00      1

The MobileNetV2 model was first trained with the base convolutional layers frozen (only the newly added dense layers were trained), which allowed it to quickly learn high-level features from the pre-trained ImageNet weights; after this initial training, the top 20 layers of the base model were unfrozen and fine-tuned using a very small learning rate (1e-5) with the Adam optimizer to carefully adjust deeper feature representations without destroying learned knowledge. During initial training, the model rapidly improved and reached very high validation accuracy (~0.99+), and after fine-tuning it achieved near-perfect performance. Finally, on the test set, the model achieved an accuracy of about 0.9993, with almost all classes showing precision, recall, and F1-scores of 1.00, indicating extremely strong generalization and minimal classification errors.


Overall Performance Comparison

This line prints a final comparison of all three models—Baseline, Deep CNN, and MobileNetV2—by displaying their accuracy and execution/training time side by side. It helps you directly evaluate how each model performs in terms of correctness (accuracy) and computational cost (time taken). From your results, the Baseline model achieves very high accuracy but with a certain processing time, the Deep model has lower accuracy (around 0.78) with its own training time, while MobileNetV2 stands out with extremely high accuracy (around 0.999) along with its training time, making it the most effective model overall in terms of performance and efficiency balance.


In [45]:
print("Baseline:", baseline_acc, baseline_time)
print("Deep:", deep_acc, deep_time)
print("MobileNetV2:", acc, mn_time)

Baseline: 0.9826666712760925 2671.8814692497253
Deep: 0.7806666493415833 2914.298819065094
MobileNetV2: 0.9993333220481873 1932.579525232315


This output compares the performance and efficiency of three models: the Baseline model achieves a high accuracy of about 0.983 but takes around 2672 seconds to run, the Deep CNN model performs worse with an accuracy of about 0.781 and also takes the longest time at about 2914 seconds, while the MobileNetV2 model clearly outperforms both by achieving an almost perfect accuracy of about 0.999 with the shortest training time of about 1933 seconds, showing that transfer learning with MobileNetV2 provides the best balance of accuracy and computational efficiency among the three.
